In [ ]:
import pandas as pd
import pathlib
import glob
import os
import numpy as np
import warnings
import pyanalib.split_df_helpers as splh

# 1. Suppress the annoying warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from tables import NaturalNameWarning
warnings.filterwarnings('ignore', category=NaturalNameWarning)
import os, sys, pathlib, glob
import pandas as pd
import numpy as np
import pyanalib.split_df_helpers as splh

def consolidate_buffered(folder_path, output_name, split_margin_gb=1.0):
    """
    Consolidates files by buffering them in memory and saving to a new split 
    whenever the buffer exceeds split_margin_gb.
    Removes initial files upon successful completion.
    """
    out_dir = pathlib.Path("/exp/sbnd/data/users/lpelegri/cafpyana_data")
    out_dir.mkdir(parents=True, exist_ok=True)
    output_file = out_dir / f"{output_name}.df"

    input_folder = pathlib.Path(folder_path)
    files = sorted(glob.glob(str(input_folder / f"*{output_name}*.df")))
    
    if not files:
        print(f"No files found for {output_name}")
        return

    # 1. Discover keys
    with pd.HDFStore(files[0], mode='r') as store:
        keys2load = [k.lstrip('/').rsplit('_', 1)[0] for k in store.keys() if 'split' not in k]
        keys2load = sorted(list(set(keys2load)))

    df_buffers = {k: [] for k in keys2load}
    size_counters = {k: 0 for k in keys2load}
    k_idx = 0 
    total_ntuples_seen = 0 

    print(f"Starting buffered consolidation for {len(files)} files...")

    # Use context manager to ensure the output file is closed properly before deletion starts
    with pd.HDFStore(output_file, mode='w') as hdf_out:
        for file_path in files:
            print(f"Reading {os.path.basename(file_path)}...")
            file_data = splh.load_dfs(file_path, keys2load)
            
            current_file_ntuples = 0
            for df in file_data.values():
                if df is not None:
                    current_file_ntuples = len(df.index.get_level_values("__ntuple").unique())
                    break

            for k in keys2load:
                df = file_data[k]
                if df is not None:
                    # Update __ntuple levels with global offset
                    old_ntuple_level = df.index.get_level_values("__ntuple")
                    new_ntuple_level = old_ntuple_level + total_ntuples_seen
                    df.index = df.index.set_levels(new_ntuple_level.unique(), level="__ntuple")

                    size_gb = df.memory_usage(deep=True).sum() / (1024**3)
                    size_counters[k] += size_gb
                    df_buffers[k].append(df)
            
            total_ntuples_seen += current_file_ntuples
            
            if any(val >= split_margin_gb for val in size_counters.values()):
                print(f"--- Buffer limit reached. Flushing to split_{k_idx} ---")
                for k in keys2load:
                    if df_buffers[k]:
                        concat_df = pd.concat(df_buffers[k], ignore_index=False)
                        hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
                        df_buffers[k] = []
                        size_counters[k] = 0
                k_idx += 1
            
            del file_data

        if any(len(b) > 0 for b in df_buffers.values()):
            print(f"--- Final Flush: split_{k_idx} ---")
            for k in keys2load:
                if df_buffers[k]:
                    concat_df = pd.concat(df_buffers[k], ignore_index=False)
                    hdf_out.put(key=f"{k}_{k_idx}", value=concat_df, format="fixed")
            k_idx += 1

        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [k_idx]}), format="fixed")

    print(f"\nConsolidation finished. Total splits created: {k_idx}")

    # --- CLEANUP: Remove initial files ---
    print(f"Cleaning up {len(files)} initial files...")
    for file_path in files:
        try:
            os.remove(file_path)
        except OSError as e:
            print(f"Error deleting {file_path}: {e}")

    print("Cleanup Complete!")
    return output_file

file_list = ["cc1pi_5e18_CV"]

# --- Usage ---
SOURCE = "/exp/sbnd/data/users/lpelegri/cafpyana_data_transfer_folder"
NAME = file_list[0]

consolidate_buffered(SOURCE, NAME)